In [2]:
!git clone https://github.com/MR-just01/Llama3.2-Reasoning

fatal: destination path 'Llama3.2-Reasoning' already exists and is not an empty directory.


In [3]:
%cd Llama3.2-Reasoning

/kaggle/working/Llama3.2-Reasoning


In [4]:
import pandas as pd

df = pd.read_csv("data/processed/reasoning_dataset (1).csv")
print(df.shape)
df.head()

(30193, 6)


,instruction,input,reasoning,answer,dataset,task_type
0,Solve the following math reasoning problem ste...,Nicole collected 400 Pokemon cards. Cindy coll...,Cindy has 400 x 2 = <<400*2=800>>800 cards.\nN...,150,gsm8k,math_reasoning
1,Solve the following math reasoning problem ste...,James had two browsers on his computer. In eac...,The total number of tabs in three windows of e...,60,gsm8k,math_reasoning
2,Solve the following multiple-choice math reaso...,The 100-milliliter solution of sugar and water...,In the original solution the amount of sugar i...,50,AQUA-RAT,math_reasoning
3,Solve the following multiple-choice math reaso...,"If 0.75 : x :: 5 : 8, then x is equal to:\n\nC...",Explanation:\n(x x 5) = (0.75 x 8)\nx=6/5\n=1....,1.2,AQUA-RAT,math_reasoning
4,Solve the following multiple-choice math reaso...,The price of Darjeeling tea (in rupees per kil...,Explanation :\nPrice of Darjeeling tea (in rup...,May 20,AQUA-RAT,math_reasoning


In [5]:
!pip install -q \
transformers==4.56.2 \
trl==0.19.1 \
peft==0.17.1 \
bitsandbytes==0.47.0 \
accelerate \
datasets \
sentencepiece \
scikit-learn

In [6]:
import transformers
import trl
import peft
import torch

print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)
print("Torch:", torch.__version__)

Transformers: 4.56.2
TRL: 0.19.1
PEFT: 0.17.1
Torch: 2.10.0+cu128


In [9]:
import os
print(os.listdir("/kaggle/working"))

['.virtual_documents', 'Llama3.2-Reasoning']


In [10]:
import transformers

print(transformers.__version__)
print(transformers.__file__)

!pip show transformers

4.56.2
/usr/local/lib/python3.12/dist-packages/transformers/__init__.py
Name: transformers
Version: 4.56.2
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.12/dist-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, requests, safetensors, tokenizers, tqdm
Required-by: kaggle-environments, peft, sentence-transformers, trl


In [11]:
from transformers import AutoTokenizer, AutoConfig

MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)



config = AutoConfig.from_pretrained(MODEL_NAME)

print(config.model_type)

llama


In [12]:
def format_prompt(row):

    messages = [
        {
            "role": "user",
            "content":
                f"{row['instruction']}\n\n{row['input']}"
        },
        {
            "role": "assistant",
            "content":
                f"Reasoning:\n{row['reasoning']}\n\n"
                f"Final Answer:\n{row['answer']}"
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

In [13]:
df["text"] = df.apply(format_prompt, axis=1)

In [14]:
print(df["text"].iloc[0])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 07 Aug 2026

<|eot_id|><|start_header_id|>user<|end_header_id|>

Solve the following math reasoning problem step by step.

Nicole collected 400 Pokemon cards. Cindy collected twice as many, and Rex collected half of Nicole and Cindy's combined total. If Rex divided his card equally among himself and his three younger siblings, how many cards does Rex have left?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Reasoning:
Cindy has 400 x 2 = <<400*2=800>>800 cards.
Nicole and Cindy have 400 + 800 = <<400+800=1200>>1200 cards.
Rex has 1200/2 = <<1200/2=600>>600 cards.
Rex is left with 600/(3+1=4) = <<600/4=150>>150 cards

Final Answer:
150<|eot_id|>


In [15]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    random_state=42,
    shuffle=True
)

In [16]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(
    train_df,
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_df,
    preserve_index=False
)

In [17]:
print(train_dataset)
print(train_dataset.features)
print(train_dataset[0])

Dataset({
    features: ['instruction', 'input', 'reasoning', 'answer', 'dataset', 'task_type', 'text'],
    num_rows: 27173
})
{'instruction': Value('string'), 'input': Value('string'), 'reasoning': Value('string'), 'answer': Value('string'), 'dataset': Value('string'), 'task_type': Value('string'), 'text': Value('string')}
{'instruction': 'Solve the following math reasoning problem step by step.', 'input': 'Jennifer decides to share her sweets between herself and her 3 friends. She has 212 green sweets, 310 blue sweets and 502 yellow sweets. How many sweets will Jennifer and her friends get each?', 'reasoning': 'Jennifer has a total of 212 + 310 + 502 = <<212+310+502=1024>>1,024 sweets.\nJennifer and her friends will get 1,024/4 = <<1024/4=256>>256 sweets each.', 'answer': '256', 'dataset': 'gsm8k', 'task_type': 'math_reasoning', 'text': '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 07 Aug 2026\n\n<|eot_id|><|start_

In [18]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [19]:
model.config.use_cache = False
model.enable_input_require_grads()

# **k-bit training**

In [20]:
import trl
print(trl.__version__)

0.19.1


In [21]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

In [22]:
print(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((3072,)

## **Configure LORA**

In [23]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

In [24]:
from peft import get_peft_model

model = get_peft_model(model, lora_config)

In [25]:
model.print_trainable_parameters()

trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


In [26]:
from trl import SFTConfig

training_args = SFTConfig(
    # Output
    output_dir="./llama3_reasoning",

    # Training
    num_train_epochs=1,
    learning_rate=2e-4,

    # Batching
    per_device_train_batch_size=1,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,

    # Precision
    fp16=True,
    bf16=False,

    # Optimizer
    optim="paged_adamw_8bit",

    # Scheduler
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,

    # Logging
    logging_steps=5,
    report_to="none",

    # Checkpoints
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    eval_strategy="no",

    # Dataset
    dataset_text_field="text",
    max_length=1024,
    packing=False,

    # Misc
    remove_unused_columns=False,
    seed=42,

    max_grad_norm=0.3
)

In [27]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

Adding EOS to train dataset:   0%|          | 0/27173 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/27173 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/27173 [00:00<?, ? examples/s]

In [28]:
import inspect
from trl import SFTTrainer

print(inspect.signature(SFTTrainer))

(model: Union[str, torch.nn.modules.module.Module, transformers.modeling_utils.PreTrainedModel], args: Union[trl.trainer.sft_config.SFTConfig, transformers.training_args.TrainingArguments, NoneType] = None, data_collator: Optional[transformers.data.data_collator.DataCollator] = None, train_dataset: Union[datasets.arrow_dataset.Dataset, datasets.iterable_dataset.IterableDataset, NoneType] = None, eval_dataset: Union[datasets.arrow_dataset.Dataset, dict[str, datasets.arrow_dataset.Dataset], NoneType] = None, processing_class: Union[transformers.tokenization_utils_base.PreTrainedTokenizerBase, transformers.image_processing_utils.BaseImageProcessor, transformers.feature_extraction_utils.FeatureExtractionMixin, transformers.processing_utils.ProcessorMixin, NoneType] = None, compute_loss_func: Optional[Callable] = None, compute_metrics: Optional[Callable[[transformers.trainer_utils.EvalPrediction], dict]] = None, callbacks: Optional[list[transformers.trainer_callback.TrainerCallback]] = None

In [29]:
import math

print("="*40)

print("Training Samples:",
      len(train_dataset))

print("Validation Samples:",
      len(val_dataset))

steps = math.ceil(
    len(train_dataset) /
    (
        training_args.per_device_train_batch_size *
        training_args.gradient_accumulation_steps
    )
)

print("Steps per Epoch:",
      steps)

print("="*40)

token_lengths = df["text"].apply(
    lambda x: len(
        tokenizer(x)["input_ids"]
    )
)

print(token_lengths.describe())

print("Maximum Tokens:",
      token_lengths.max())

Training Samples: 27173
Validation Samples: 3020
Steps per Epoch: 6794
count    30193.000000
mean       205.107210
std         65.741242
min         80.000000
25%        162.000000
50%        195.000000
75%        237.000000
max        857.000000
Name: text, dtype: float64
Maximum Tokens: 857


In [30]:
import trl
print(trl.__version__)

import transformers
print(transformers.__version__)

import peft
print(peft.__version__)

0.19.1
4.56.2
0.17.1


In [31]:
batch = next(iter(trainer.get_train_dataloader()))

for k, v in batch.items():
    print(k, v.shape)

input_ids torch.Size([2, 220])
attention_mask torch.Size([2, 220])
labels torch.Size([2, 220])


In [32]:
import torch
print(model.hf_device_map)
print(torch.cuda.device_count())

{'': 0}
2


In [33]:
batch = next(iter(trainer.get_train_dataloader()))

for k, v in batch.items():
    print(k, v.shape)

outputs = model(
    input_ids=batch["input_ids"].to(model.device),
    attention_mask=batch["attention_mask"].to(model.device),
    labels=batch["labels"].to(model.device),
)

print(outputs.loss)
print(outputs.logits.shape)

input_ids torch.Size([2, 220])
attention_mask torch.Size([2, 220])
labels torch.Size([2, 220])
tensor(3.0609, device='cuda:0', grad_fn=<NllLossBackward0>)
torch.Size([2, 220, 128256])


In [34]:
print(training_args)

SFTConfig(
_n_gpu=2,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
activation_offloading=False,
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
assistant_only_loss=False,
auto_find_batch_size=False,
average_tokens_across_devices=False,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
chat_template_path=None,
completion_only_loss=None,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
dataset_kwargs=None,
dataset_num_proc=None,
dataset_text_field=text,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=False,
do_predict=False,
do_train=False,
eos_tok

In [35]:
print(model.is_gradient_checkpointing)

print(model.config.use_cache)

print(model.training)

print(type(model))

True
False
True
<class 'peft.peft_model.PeftModelForCausalLM'>


In [36]:
print(model.device)
print(next(model.parameters()).device)

cuda:0
cuda:0


In [39]:
import os

print(os.listdir("/kaggle/working/Llama3.2-Reasoning/llama3_reasoning"))

['checkpoint-1500', 'checkpoint-2000', 'README.md']


In [41]:
from transformers.trainer_utils import get_last_checkpoint

checkpoint = get_last_checkpoint("/kaggle/working/Llama3.2-Reasoning/llama3_reasoning")

trainer.train(resume_from_checkpoint=checkpoint)

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
3005,0.397000
3010,0.458900
3015,0.392100
3020,0.563100
3025,0.378800
3030,0.423200
3035,0.392500
3040,0.374000
3045,0.324800
3050,0.424300


TrainOutput(global_step=3396, training_loss=0.04718088022530851, metrics={'train_runtime': 2345.1364, 'train_samples_per_second': 11.587, 'train_steps_per_second': 1.449, 'total_flos': 1.1116293285638554e+17, 'train_loss': 0.04718088022530851})

In [42]:
OUTPUT_DIR = "/kaggle/working/Llama3.2-Reasoning/final_model"

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Model saved successfully!")

Model saved successfully!


In [43]:
import os

print(os.listdir(OUTPUT_DIR))

['special_tokens_map.json', 'adapter_model.safetensors', 'tokenizer.json', 'training_args.bin', 'adapter_config.json', 'tokenizer_config.json', 'chat_template.jinja', 'README.md']


In [ ]:
1 import os

 print(os.listdir("/kaggle/working/Llama3.2-Reasoning/llama3_reasoning"))

In [ ]:
OUTPUT_DIR = "/kaggle/working/Llama3.2-Reasoning"

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Model saved successfully!")

In [47]:
import os

print(os.listdir("/kaggle/working"))

['final_model.zip', '.virtual_documents', 'Llama3.2-Reasoning']


In [48]:
import os

print(os.listdir("/kaggle/working/Llama3.2-Reasoning"))

['04_llama_lora_training.ipynb', '.git', 'llama3_reasoning', 'final_model', 'LICENSE', 'notebooks', 'src', 'reports', 'outputs', 'data', '.gitignore', 'README.md', 'requirements.txt']


In [50]:
import os
print(round(os.path.getsize("/kaggle/working/final_model.zip")/(1024*1024),2))

88.48


In [49]:
import os

for root, dirs, files in os.walk("/kaggle/working"):
    print(root)
    for f in files:
        print("   ", f)

/kaggle/working
    final_model.zip
/kaggle/working/.virtual_documents
    __notebook_source__.ipynb
/kaggle/working/Llama3.2-Reasoning
    04_llama_lora_training.ipynb
    LICENSE
    .gitignore
    README.md
    requirements.txt
/kaggle/working/Llama3.2-Reasoning/.git
    config
    HEAD
    description
    packed-refs
    index
/kaggle/working/Llama3.2-Reasoning/.git/info
    exclude
/kaggle/working/Llama3.2-Reasoning/.git/branches
/kaggle/working/Llama3.2-Reasoning/.git/objects
/kaggle/working/Llama3.2-Reasoning/.git/objects/info
/kaggle/working/Llama3.2-Reasoning/.git/objects/pack
    pack-fc9b0d8136c6e261b4260de0697f931eff2141cf.pack
    pack-fc9b0d8136c6e261b4260de0697f931eff2141cf.idx
/kaggle/working/Llama3.2-Reasoning/.git/refs
/kaggle/working/Llama3.2-Reasoning/.git/refs/heads
    main
/kaggle/working/Llama3.2-Reasoning/.git/refs/tags
/kaggle/working/Llama3.2-Reasoning/.git/refs/remotes
/kaggle/working/Llama3.2-Reasoning/.git/refs/remotes/origin
    HEAD
/kaggle/working/Llama

In [44]:
import shutil

shutil.make_archive(
    "/kaggle/working/final_model",
    "zip",
    OUTPUT_DIR
)

'/kaggle/working/final_model.zip'

In [46]:
from IPython.display import FileLink

FileLink("/kaggle/working/final_model.zip")

/kaggle/working/final_model.zip

In [45]:
print(type(model))

<class 'peft.peft_model.PeftModelForCausalLM'>
